# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes 
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [1]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


 #Answer 
 
The continent with the highest growth index in the year 2022 is also the one that has seen the greatest relative growth since the year 1970. Europe has grown much slower than itself—Africa is set to lead the way in relative terms.T

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*

A line chart is best to show how the population of the six continents has changed over time. It's an index with 1970 set at 100 for each continent so that you can compare the growth of each continent given very different starting points.

In [2]:
#aggregate population by continent and year
continent_pop = (
    df_long.groupby(['Continent', 'Year'])['Population']
    .sum()
    .reset_index()
)

# Calculate relative growth (1970 = 100)
baseline = (
    continent_pop[continent_pop['Year'] == 1970]
    .set_index('Continent')['Population']
)

continent_pop['Growth Index'] = (
    continent_pop.apply(
        lambda row: row['Population'] / baseline[row['Continent']] * 100,
        axis=1
    )
)

# Identify fastest-growing continent
latest = continent_pop[continent_pop['Year'] == 2022]
fastest = latest.loc[latest['Growth Index'].idxmax()]

print("Fastest-growing continent:", fastest['Continent'])
print("Growth since 1970:",
      round(fastest['Growth Index'] - 100, 1), "%")

# Create line chart
fig = px.line(
    continent_pop,
    x='Year',
    y='Growth Index',
    color='Continent',
    markers=True,
    title='Africa has experienced the fastest relative population growth since 1970',
    labels={
        'Year': 'Year',
        'Growth Index': 'Population index (1970 = 100)',
        'Continent': 'Continent'
    },
    color_discrete_sequence=px.colors.qualitative.Safe,
    template='plotly_white'
)

fig.update_layout(
    title_x=0.03,
    legend_title_text='',
    hovermode='x unified',
    yaxis_showgrid=False,
    xaxis_showgrid=False
)

fig.update_yaxes(rangemode='tozero')
fig.show()

Fastest-growing continent: Africa
Growth since 1970: 290.4 %


---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

15 countries with largest population increase in former absolute number. Print answers: How many of these are African countries? Absolute increases are greatest in countries that already have large or rapidly growing populations.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*

A sorted horizontal bar chart is ideal to rank the 15 countries with the largest absolute population growth. Horizontal bars allow easy identification of country names, while color differentiates African from non-African countries.

In [3]:
# Q2 — Population increase between 1970 and 2022

df['Population Increase'] = (
    df['2022 Population'] - df['1970 Population']
)

# Select top 15 countries
top15 = (
    df.nlargest(15, 'Population Increase')
    .sort_values('Population Increase', ascending=True)
    .copy()
)

# Count African countries
africa_count = (top15['Continent'] == 'Africa').sum()

print("Number of African countries in the top 15:", africa_count)

# Colour African countries differently
top15['Region'] = np.where(
    top15['Continent'] == 'Africa',
    'Africa',
    'Other continents'
)

fig = px.bar(
    top15,
    x='Population Increase',
    y='Country',
    orientation='h',
    color='Region',
    text=top15['Population Increase'].apply(
        lambda x: f'{x / 1e6:.1f}M'
    ),
    title='A small group of populous countries accounts for the largest population gains',
    labels={
        'Population Increase': 'Additional people since 1970',
        'Country': '',
        'Region': ''
    },
    color_discrete_map={
        'Africa': '#E69F00',
        'Other continents': '#7A8793'
    },
    template='plotly_white'
)

fig.update_traces(
    textposition='outside',
    cliponaxis=False
)

fig.update_layout(
    title_x=0.03,
    showlegend=True,
    legend_title_text='',
    xaxis_showgrid=False,
    yaxis_showgrid=False
)

fig.update_xaxes(tickformat='.2s')
fig.show()

Number of African countries in the top 15: 4


---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

The largest countries are not necessarily populated countries. India and Bangladesh, for example, are very high in population to land ratio and Russia, Canada, and Australia in land to population density. This demonstrates that population size is not solely based on land area.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*

A scatter plot is a good technique here to show the relationship between two numeric amounts, land area of a country and population. Logarithmic axs help you quickly identify countries of widely varying sizes, and coloring by continent provides patterns and potential outliers in each region.

In [4]:
# Q3 — Land area versus population

fig = px.scatter(
    df,
    x='Area',
    y='2022 Population',
    color='Continent',
    hover_name='Country',
    hover_data={
        'Area': ':,',
        '2022 Population': ':,',
        'Density': ':.1f'
    },
    log_x=True,
    log_y=True,
    title='Land area alone does not explain which countries have the largest populations',
    labels={
        'Area': 'Land area (km², logarithmic scale)',
        '2022 Population': 'Population in 2022 (logarithmic scale)',
        'Continent': 'Continent'
    },
    color_discrete_sequence=px.colors.qualitative.Safe,
    template='plotly_white'
)

# Highlight notable countries
outliers = df[
    df['Country'].isin([
        'India', 'Bangladesh', 'China',
        'Russia', 'Canada', 'Australia'
    ])
]

fig.add_scatter(
    x=outliers['Area'],
    y=outliers['2022 Population'],
    mode='text',
    text=outliers['Country'],
    textposition='top center',
    showlegend=False,
    textfont=dict(size=10, color='black')
)

fig.update_traces(
    marker=dict(size=10, opacity=0.75),
    selector=dict(mode='markers')
)

fig.update_layout(
    title_x=0.03,
    legend_title_text='',
    xaxis_showgrid=False,
    yaxis_showgrid=False
)

fig.show()

---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

Population density differs substantially across continents and between countries within the same continent. The logarithmic scale is useful because a few extremely dense countries can otherwise dominate the visualisation. The box plots reveal both typical density and the variation around it.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*

It is appropriate to use a box plot, as we are being asked to compare the variation within and between continents rather than just the average density. It makes it easier to compare differences in population density because it shows the median, interquartile range, and outliers.

In [6]:


fig = px.box(
    df,
    x='Continent',
    y='Density',
    color='Continent',
    points='outliers',
    title='Population density varies dramatically within each continent',
    labels={
        'Continent': '',
        'Density': 'People per km² (2022)'
    },
    color_discrete_sequence=px.colors.qualitative.Safe,
    template='plotly_white'
)

fig.update_traces(
    marker=dict(size=5, opacity=0.65),
    line=dict(width=1.5)
)

fig.update_layout(
    title_x=0.03,
    showlegend=False,
    xaxis_showgrid=False,
    yaxis_showgrid=False
)

fig.update_yaxes(type='log')

fig.show()

---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

Answer#
The map highlights geographical differences in population growth. Countries experiencing positive population change are represented on one side of the diverging colour scale, while countries experiencing decline appear on the other. This helps identify areas of demographic expansion and contraction.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.



*YOUR ANSWER HERE*

A choropleth map is the best choice because the question is explicitly geographical. It uses colour to show percentage population change between 2000 and 2022, allowing us to identify regional patterns and countries experiencing population decline.

In [7]:
# Q5 — Percentage population change between 2000 and 2022

df['Population Change (%)'] = (
    (df['2022 Population'] - df['2000 Population'])
    / df['2000 Population'] * 100
)

# Count growing and shrinking countries
growing = (df['Population Change (%)'] > 0).sum()
shrinking = (df['Population Change (%)'] < 0).sum()

print("Countries with population growth:", growing)
print("Countries with population decline:", shrinking)

# Symmetrical colour range around zero
limit = df['Population Change (%)'].abs().max()

fig = px.choropleth(
    df,
    locations='CCA3',
    locationmode='ISO-3',
    color='Population Change (%)',
    hover_name='Country',
    hover_data={
        'Continent': True,
        'Population Change (%)': ':.1f',
        'CCA3': False
    },
    color_continuous_scale='PuOr',
    range_color=[-limit, limit],
    color_continuous_midpoint=0,
    title='Population growth is widespread, but several countries experienced decline',
    labels={
        'Population Change (%)': 'Population change (%)'
    },
    template='plotly_white'
)

fig.update_layout(
    title_x=0.03,
    geo=dict(
        showframe=False,
        showcoastlines=True,
        coastlinecolor='lightgrey',
        showland=True,
        landcolor='#F2F2F2',
        showocean=True,
        oceancolor='white'
    ),
    coloraxis_colorbar=dict(
        title='Change (%)'
    )
)

fig.show()

Countries with population growth: 200
Countries with population decline: 34


---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

Asia occupies the largest area of the treemap, reflecting its dominant share of the global population. The nested rectangles show how individual countries contribute to their respective continents. China and India are particularly important contributors to the global population.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*

A treemap is ideal for hierarchical part-to-whole data. It shows the contribution of each continent and the countries within it using nested rectangles, with rectangle size proportional to population.

In [8]:
# Q6 — World population distribution in 2022

world_total = df['2022 Population'].sum()

print("Total world population:",
      f"{world_total / 1e9:.2f} billion")

# Calculate continent population shares
continent_totals = (
    df.groupby('Continent')['2022 Population']
    .sum()
    .sort_values(ascending=False)
)

print("\nPopulation share by continent:")
print((continent_totals / world_total * 100).round(1))

fig = px.treemap(
    df,
    path=['Continent', 'Country'],
    values='2022 Population',
    color='Continent',
    color_discrete_sequence=px.colors.qualitative.Safe,
    hover_name='Country',
    hover_data={
        '2022 Population': ':,'
    },
    title='Asia contains the largest share of the world’s population',
    labels={
        '2022 Population': 'Population in 2022'
    }
)

fig.update_traces(
    textinfo='label+percent parent',
    textfont=dict(size=12),
    marker=dict(line=dict(width=1, color='white'))
)

fig.update_layout(
    template='plotly_white',
    title_x=0.03,
    margin=dict(t=70, l=10, r=10, b=10)
)

fig.show()

Total world population: 7.97 billion

Population share by continent:
Continent
Asia             59.2
Africa           17.9
Europe            9.3
North America     7.5
South America     5.5
Oceania           0.6
Name: 2022 Population, dtype: float64


---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

Africa's share of the world's population has increased considerably since 1970, while Europe's share has declined. Asia continues to account for the largest proportion of the global population, although its relative share has changed over time. The rest of the world represents North America, South America and Oceania combined.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*

A 100% stacked bar chart is appropriate because the question concerns changes in proportions rather than absolute population. Each bar represents 100% of the world population in a given year, making it easy to compare the changing shares of Africa, Asia, Europe and the rest of the world.

In [9]:
# Q7 — World population shares by continent over time

# Group the three remaining continents
df_long['Region Group'] = np.where(
    df_long['Continent'].isin(['Africa', 'Asia', 'Europe']),
    df_long['Continent'],
    'Rest of the world'
)

# Aggregate population by year and region
shares = (
    df_long.groupby(['Year', 'Region Group'])['Population']
    .sum()
    .reset_index()
)

# Calculate world population share
shares['World Share (%)'] = (
    shares['Population']
    / shares.groupby('Year')['Population'].transform('sum')
    * 100
)

# Set order of categories
region_order = [
    'Asia',
    'Africa',
    'Europe',
    'Rest of the world'
]

shares['Region Group'] = pd.Categorical(
    shares['Region Group'],
    categories=region_order,
    ordered=True
)

shares = shares.sort_values(['Year', 'Region Group'])

# Calculate change in share between 1970 and 2022
share_comparison = shares.pivot(
    index='Region Group',
    columns='Year',
    values='World Share (%)'
)

share_comparison['Change (percentage points)'] = (
    share_comparison[2022] - share_comparison[1970]
)

print("Change in world population share (1970–2022):")
print(share_comparison[[1970, 2022, 'Change (percentage points)']].round(2))

fig = px.bar(
    shares,
    x='Year',
    y='World Share (%)',
    color='Region Group',
    barmode='stack',
    title='Africa’s share is rising while Europe’s share of world population is declining',
    category_orders={'Region Group': region_order},
    color_discrete_map={
        'Asia': '#0072B2',
        'Africa': '#E69F00',
        'Europe': '#999999',
        'Rest of the world': '#CC79A7'
    },
    labels={
        'Year': 'Year',
        'World Share (%)': 'Share of world population (%)',
        'Region Group': ''
    },
    template='plotly_white'
)

fig.update_layout(
    title_x=0.03,
    legend_title_text='',
    yaxis_range=[0, 100],
    xaxis_showgrid=False,
    yaxis_showgrid=False,
    hovermode='x unified'
)

fig.update_traces(
    marker=dict(line=dict(width=0.5, color='white'))
)

fig.show()

Change in world population share (1970–2022):
Year                1970   2022  Change (percentage points)
Region Group                                               
Asia               58.06  59.21                        1.15
Africa              9.89  17.89                        8.00
Europe             17.76   9.32                       -8.44
Rest of the world  14.29  13.57                       -0.72
